In [1]:
import pandas as pd
import geopandas as gpd

# 1. Carrega o shapefile das células de Voronoi (UTM SIRGAS 2000 23S - EPSG:31983)
mapa_areas = gpd.read_file(r'C:\Users\A0050637\Documents\01_Diversos\FIAP\GS_2026_2sem\shapefiles_utm_31983\celulas_voronoi_utm.shp')
if mapa_areas.crs is None:
    mapa_areas = mapa_areas.set_crs(epsg=31983)

# 2. Carrega as bases em Excel
base_bo = pd.read_excel(r"C:\Users\A0050637\Documents\01_Diversos\FIAP\GS_2026_2sem\base_evento\BO_SP.xlsx")
base_acidente = pd.read_excel(r"C:\Users\A0050637\Documents\01_Diversos\FIAP\GS_2026_2sem\base_evento\ACIDENTE_TRANSITO_SP.xlsx")
base_alagamento = pd.read_excel(r"C:\Users\A0050637\Documents\01_Diversos\FIAP\GS_2026_2sem\base_evento\ALAGAMENTO_SP.xlsx")

# 3. Função para ler shapefiles de risco, converter para WGS84 (graus) e extrair centroide
def preparar_base_risco(caminho_shp, grupo):
    gdf = gpd.read_file(caminho_shp).to_crs(epsg=4326)
    df = pd.DataFrame(gdf.drop(columns='geometry'))
    df['LATITUDE'] = gdf.geometry.centroid.y
    df['LONGITUDE'] = gdf.geometry.centroid.x
    df['GRUPO_ANALISE'] = grupo
    return df

base_inundacao = preparar_base_risco(
    r'C:\Users\A0050637\Documents\01_Diversos\FIAP\GS_2026_2sem\base_evento\risco_ocorrencia_inundacao_v2\risco_ocorrencia_inundacao_v2.shp',
    'INUNDACAO'
)
base_risco_alagamento = preparar_base_risco(
    r'C:\Users\A0050637\Documents\01_Diversos\FIAP\GS_2026_2sem\base_evento\risco_ocorrencia_alagamento_v2\risco_ocorrencia_alagamento_v2.shp',
    'ALAGAMENTO_PONTO'
)
base_deslizamento = preparar_base_risco(
    r'C:\Users\A0050637\Documents\01_Diversos\FIAP\GS_2026_2sem\base_evento\risco_ocorrencia_deslizamento_v2\risco_ocorrencia_deslizamento_v2.shp',
    'DESLIZAMENTO'
)

# Empilha as 3 bases de risco climático (equivalente ao rbind do R)
base_eventos_clima = pd.concat([base_inundacao, base_risco_alagamento, base_deslizamento], ignore_index=True)

# 4. Função de Spatial Join com limpeza de coordenadas inválidas
def vincular_voronoi(df, gdf_voronoi):
    df_valido = df.copy()
    
    # Converte vírgula para ponto decimal e transforma valores inválidos (ex: '-') em NaN
    for col in ['LATITUDE', 'LONGITUDE']:
        df_valido[col] = (
            df_valido[col]
            .astype(str)
            .str.replace(',', '.', regex=False)
            .str.strip()
        )
        df_valido[col] = pd.to_numeric(df_valido[col], errors='coerce')
    
    # Remove registros sem coordenadas válidas
    df_valido = df_valido.dropna(subset=['LATITUDE', 'LONGITUDE'])
    
    # Cria geometria de pontos em WGS84 (EPSG:4326) e reprojeta para o CRS do Voronoi (UTM 31983)
    gdf_pontos = gpd.GeoDataFrame(
        df_valido,
        geometry=gpd.points_from_xy(df_valido['LONGITUDE'], df_valido['LATITUDE']),
        crs='EPSG:4326'
    ).to_crs(gdf_voronoi.crs)
    
    # Realiza o Spatial Join (verifica em qual polígono cada ponto cai)
    resultado = gpd.sjoin(gdf_pontos, gdf_voronoi, how='left', predicate='within')
    
    # Remove colunas técnicas de geometria/índice e retorna um DataFrame Pandas puro
    resultado = pd.DataFrame(resultado.drop(columns=['geometry', 'index_right'], errors='ignore'))
    return resultado

# 5. Aplica o Spatial Join nas 4 bases
base_bo = vincular_voronoi(base_bo, mapa_areas)
base_acidente = vincular_voronoi(base_acidente, mapa_areas)
base_alagamento = vincular_voronoi(base_alagamento, mapa_areas)
base_eventos_clima = vincular_voronoi(base_eventos_clima, mapa_areas)

print("Processamento finalizado com sucesso!")



C:\Users\A0050637\AppData\Local\Temp\ipykernel_30840\1018352152.py:18: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  df['LATITUDE'] = gdf.geometry.centroid.y
C:\Users\A0050637\AppData\Local\Temp\ipykernel_30840\1018352152.py:19: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  df['LONGITUDE'] = gdf.geometry.centroid.x
C:\Users\A0050637\AppData\Local\Temp\ipykernel_30840\1018352152.py:18: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  df['LATITUDE'] = gdf.geometry.centroid.y
C:\Users\A0050637\AppData\Local\Temp\ipykernel_30840\1018352152.py:19: UserWarning: Geometry is in a g

Processamento finalizado com sucesso!


In [2]:
base_bo['MACRO_GRUPO'] = 'SEGURANCA_PUBLICA'
base_acidente['MACRO_GRUPO'] = 'ACIDENTE_TRANSITO'
base_alagamento['MACRO_GRUPO'] = 'ALAGAMENTO_VIAS_PUBLICAS'
base_eventos_clima['MACRO_GRUPO'] = 'ACIONAMENTO_DEFESA_CIVIL'

In [3]:
import pandas as pd


def resumir_eventos(df: pd.DataFrame) -> pd.DataFrame:
  df = df.copy()

  # 1. Converter DATA_REF para datetime e normalizar para remover horas/minutos (apenas a data)
  df["DATA_REF"] = pd.to_datetime(df["DATA_REF"]).dt.normalize()

  # 2. Calcular o total de dias corridos na base inteira
  menor_data = df["DATA_REF"].min()
  maior_data = df["DATA_REF"].max()
  total_dias = (maior_data - menor_data).days + 1

  cols = ["id_celula", "MACRO_GRUPO", "DIA_SEM", "PERIODO"]

  # 3. Agrupar contando o total de eventos E a quantidade de dias únicos com evento
  resumo = df.groupby(cols, dropna=False, as_index=False).agg(
      QUANT_EVENTO=("DATA_REF", "size"),
      DIAS_COM_EVENTO=("DATA_REF", "nunique"),
  )

  # 4. Calcular o total de dias e a probabilidade (dias únicos / total de dias)
  resumo["TOTAL_DIAS"] = total_dias
  resumo["PROB_EVENTO"] = resumo["DIAS_COM_EVENTO"] / resumo["TOTAL_DIAS"]

  # 5. Ordenar da maior probabilidade para a menor
  resumo = resumo.sort_values(by="PROB_EVENTO", ascending=False).reset_index(
      drop=True
  )

  return resumo


resumo_bo = resumir_eventos(base_bo)
resumo_acidente = resumir_eventos(base_acidente)







In [4]:
import pandas as pd


def analisar_probabilidades_completas(df: pd.DataFrame) -> pd.DataFrame:
  df = df.copy()
  df["DATA_REF"] = pd.to_datetime(df["DATA_REF"]).dt.normalize()

  # Total de dias corridos na base inteira
  total_dias = (df["DATA_REF"].max() - df["DATA_REF"].min()).days + 1

  # 1. MODELO GERAL (Por Polígono / Dia)
  geral_poligono = df.groupby("id_celula", dropna=False, as_index=False).agg(
      DIAS_COM_EVENTO_CELULA=("DATA_REF", "nunique")
  )
  geral_poligono["PROB_GERAL_DIA"] = (
      geral_poligono["DIAS_COM_EVENTO_CELULA"] / total_dias
  )

  # 2. MODELO DETALHADO (Por Polígono, Macro Grupo, Dia da Semana e Período)
  cols_detalhe = ["id_celula", "MACRO_GRUPO", "DIA_SEM", "PERIODO"]
  detalhe = df.groupby(cols_detalhe, dropna=False, as_index=False).agg(
      DIAS_COM_EVENTO_PERIODO=("DATA_REF", "nunique")
  )
  detalhe["PROB_PERIODO_DIA"] = detalhe["DIAS_COM_EVENTO_PERIODO"] / total_dias

  # 3. COMBINAR OS DOIS MODELOS
  resultado_combinado = pd.merge(
      detalhe,
      geral_poligono[["id_celula", "PROB_GERAL_DIA"]],
      on="id_celula",
      how="left",
  )

  # 4. Ordenar primeiro pelos polígonos mais críticos no geral, e depois pelos piores períodos
  resultado_combinado = resultado_combinado.sort_values(
      by=["PROB_GERAL_DIA", "PROB_PERIODO_DIA"], ascending=[False, False]
  ).reset_index(drop=True)

  # 5. Selecionar e retornar apenas as colunas desejadas
  colunas_finais = [
      "id_celula",
      "MACRO_GRUPO",
      "DIA_SEM",
      "PERIODO",
      "PROB_PERIODO_DIA",
      "PROB_GERAL_DIA",
  ]
  return resultado_combinado[colunas_finais]


# Executando para suas bases
analise_bo = analisar_probabilidades_completas(base_bo)
analise_acidente = analisar_probabilidades_completas(base_acidente)


In [5]:
import pandas as pd

# 1. Criação dos DataFrames
dados_dia = {
    'DIA_SEM': ['sex', 'sáb', 'dom', 'qui', 'qua', 'seg', 'ter']
}
DIA_SEM = pd.DataFrame(dados_dia)

dados_periodo = {
    'PERIODO': ['NOITE', 'TARDE', 'MANHA', 'MADRUGADA']
}
PERIODO = pd.DataFrame(dados_periodo)

# 2. Merge cruzado (combina cada linha de DIA_SEM com todas de PERIODO)
df_combinado = pd.merge(DIA_SEM, PERIODO, how='cross')

# Cruzamento com a base principal
base_alagamento = pd.merge(base_alagamento, df_combinado, how='cross')

# 3. Seleção das colunas (com .copy() para evitar avisos do Pandas)
colunas_selecionadas = [
    "id_celula",
    "MACRO_GRUPO",
    "DIA_SEM",
    "PERIODO",
    "PROB_PERIODO_DIA",
    "PROB_GERAL_DIA"
]
analise_alagamento = base_alagamento[colunas_selecionadas].copy()

# Garantir que as colunas de probabilidade são numéricas
analise_alagamento['PROB_GERAL_DIA'] = pd.to_numeric(analise_alagamento['PROB_GERAL_DIA'], errors='coerce')
analise_alagamento['PROB_PERIODO_DIA'] = pd.to_numeric(analise_alagamento['PROB_PERIODO_DIA'], errors='coerce')

analise_alagamento["chave_analise"] = (
    analise_alagamento["id_celula"].astype(str) + "_" +
    analise_alagamento["MACRO_GRUPO"].astype(str) + "_" +
    analise_alagamento["DIA_SEM"].astype(str) + "_" +
    analise_alagamento["PERIODO"].astype(str)
)

# 4. Ordenar e remover duplicados
# Dica: incluí PROB_PERIODO_DIA como segundo critério de ordenação para desempatar
analise_alagamento = (
    analise_alagamento
    .sort_values(by=['PROB_GERAL_DIA', 'PROB_PERIODO_DIA'], ascending=[False, False])
    .drop_duplicates(subset=['chave_analise'], keep='first')
    .reset_index(drop=True)
)

analise_alagamento = analise_alagamento.drop(columns=["chave_analise"])





In [6]:
import pandas as pd

# 1. Garante que dt_ocorren está em formato de data
base_eventos_clima["dt_ocorren"] = pd.to_datetime(base_eventos_clima["dt_ocorren"])

# 2. Cria o resumo agrupado e ordenado
analise_evento_clima = (
    base_eventos_clima["id_celula"]
    .value_counts()
    .reset_index(name="qtd_eventos")
)

# 3. Calcula o intervalo global de dias e as probabilidades
intervalo_dias = (base_eventos_clima["dt_ocorren"].max() - base_eventos_clima["dt_ocorren"].min()).days
analise_evento_clima["QUANT_DIA"] = intervalo_dias

analise_evento_clima["PROB_GERAL_DIA"] = (
    analise_evento_clima["qtd_eventos"] / analise_evento_clima["QUANT_DIA"]
)
analise_evento_clima["PROB_PERIODO_DIA"] = analise_evento_clima["PROB_GERAL_DIA"]

# 4. Define o MACRO_GRUPO (ajuste o texto conforme o padrão da sua base_alagamento)
analise_evento_clima["MACRO_GRUPO"] = "DEFESA_CIVIL"

# 5. Cruza com a grade de dias e períodos (use .drop_duplicates() se 'seg' ainda estiver duplicado)
analise_evento_clima = pd.merge(analise_evento_clima, df_combinado.drop_duplicates(), how="cross")

# 6. Seleciona e ordena as colunas no layout final
colunas_selecionadas = [
    "id_celula",
    "MACRO_GRUPO",
    "DIA_SEM",
    "PERIODO",
    "PROB_PERIODO_DIA",
    "PROB_GERAL_DIA"
]
analise_evento_clima = analise_evento_clima[colunas_selecionadas]


In [7]:
import pandas as pd

# 1. Lista com os 4 DataFrames na ordem desejada
lista_dfs = [
    analise_evento_clima,
    analise_alagamento,
    analise_bo,
    analise_acidente
]

# 2. Empilha todos os DataFrames e reinicia o índice (0, 1, 2...)
probabilidades_consolidadas = pd.concat(lista_dfs, ignore_index=True)

In [8]:
# 1. Criação dos DataFrames
dados_dia = {
    'DIA_SEM': ['sex', 'sáb', 'dom', 'qui', 'qua', 'seg', 'ter']
}
DIA_SEM = pd.DataFrame(dados_dia)

dados_periodo = {
    'PERIODO': ['NOITE', 'TARDE', 'MANHA', 'MADRUGADA']
}
PERIODO = pd.DataFrame(dados_periodo)

ID_CELULA = (
    probabilidades_consolidadas[['id_celula']]
    .drop_duplicates()
    .reset_index(drop=True)
)

MACRO_GRUPO = (
    probabilidades_consolidadas[['MACRO_GRUPO']]
    .drop_duplicates()
    .reset_index(drop=True)
)

matriz_probabilidades = pd.merge(ID_CELULA, PERIODO.drop_duplicates(), how="cross")

matriz_probabilidades = pd.merge(matriz_probabilidades, DIA_SEM.drop_duplicates(), how="cross")

matriz_probabilidades = pd.merge(matriz_probabilidades, MACRO_GRUPO.drop_duplicates(), how="cross")

chaves = ['DIA_SEM', 'PERIODO', 'id_celula', 'MACRO_GRUPO']
cols = ['DIA_SEM', 'PERIODO', 'id_celula', 'MACRO_GRUPO','PROB_PERIODO_DIA', 'PROB_GERAL_DIA']

probabilidades_consolidadas = matriz_probabilidades.merge(probabilidades_consolidadas[cols], 
                                                          on=chaves, 
                                                          how='left').fillna({'PROB_PERIODO_DIA': 0, 
                                                                              'PROB_GERAL_DIA': 0})

